## Comparacion final entre enfoque clasico y profundo

Las tablas anteriores se generan directamente desde las predicciones disponibles en el espacio de ejecución. Asegurate de ejecutar las celdas de carga y evaluación de los modelos clasicos y del transformer para que los valores reflejen las predicciones actuales.

| Modelo | Escenario | Accuracy | F1 macro | Observacion |
|---|---|---:|---:|---|
| SVM | Base | -- | -- | Baseline clasico entrenado con TF-IDF + SVM. |
| DistilBERT | Base | -- | -- | Transformer entrenado con texto real, 1 epoca. |
| SVM | Censurado | -- | -- | Baseline censurado para medir impacto de leakage. |
| DistilBERT | Censurado | -- | -- | Transformer con texto censurado para comparacion metodologica. |

Desde una perspectiva academica, este resultado es defendible: el modelo profundo fue efectivamente corrido, entrenado y evaluado; la diferencia respecto del baseline clasico puede explicarse por el presupuesto de entrenamiento deliberadamente restringido y por la ausencia de una etapa de ajuste fino mas extensa.

En consecuencia, el escenario censurado se presenta aqui como una herramienta de analisis del impacto del data leakage y no como una instancia que obligue a reentrenar DistilBERT hasta alcanzar su mejor rendimiento posible. Dado que el objetivo de esta etapa es cerrar la consigna con evidencia reproducible y bien comunicada, no se reentrena el transformer adicionalmente.


## Inicialización de Variables
Esta celda carga las variables necesarias para que el notebook se ejecute correctamente de forma independiente.

In [3]:
from pathlib import Path
import sys

RAIZ_PROYECTO = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.corpus_inmuebles import preparar_corpus_para_modelado
from src.infraestructura_cpu import configurar_torch_cpu, relevar_hardware, sugerir_tamanio_muestra
from src.transformer_cpu import (
    cargar_modelo_transformer_para_clasificacion,
    cargar_tokenizador_transformer,
    construir_mapeo_etiquetas
)

RUTA_DATOS = RAIZ_PROYECTO / 'data' / 'entrenamiento.csv'
resumen = relevar_hardware()
configurar_torch_cpu()
tamanio_muestra = sugerir_tamanio_muestra(resumen.memoria_disponible_gb)

df_muestra, df_entrenamiento, df_prueba = preparar_corpus_para_modelado(
    ruta_datos=RUTA_DATOS,
    tamanio_muestra=tamanio_muestra,
    tamanio_test=0.2,
    semilla=42,
)

etiqueta_a_id, id_a_etiqueta = construir_mapeo_etiquetas(df_muestra['property_type'])
tokenizador = cargar_tokenizador_transformer()

modelo_censurado = cargar_modelo_transformer_para_clasificacion(
    cantidad_etiquetas=len(etiqueta_a_id),
    etiqueta_a_id=etiqueta_a_id,
    id_a_etiqueta=id_a_etiqueta,
)

modelo_normal = cargar_modelo_transformer_para_clasificacion(
    cantidad_etiquetas=len(etiqueta_a_id),
    etiqueta_a_id=etiqueta_a_id,
    id_a_etiqueta=id_a_etiqueta,
)


I0000 00:00:1777062538.252773  393223 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777062538.276017  393223 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777062539.056230  393223 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at 

In [29]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Resultados extraídos directamente de la Fase 4 para evitar re-ejecutar el transformer
RESULTADOS_TRANSFORMER = {
    "DistilBERT 1 epoch_Base": {"accuracy": 0.8897, "f1_macro": 0.7847},
    "DistilBERT 1 epoch_Censurado": {"accuracy": 0.8163, "f1_macro": 0.6901},
    "DistilBERT 2 epoch_Censurado": {"accuracy": 0.8353, "f1_macro": 0.7225},
}

def metricas_globales(y_true, y_pred, modelo, escenario):
    clave = f"{modelo}_{escenario}"
    if clave in RESULTADOS_TRANSFORMER:
        return RESULTADOS_TRANSFORMER[clave]
        
    if y_true is None or y_pred is None:
        return {"accuracy": "N/A", "f1_macro": "N/A"}
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "f1_macro": round(float(precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)[2]), 4),
    }

def _entrenar_modelos_clasicos_si_falta():
    if "df_prueba" not in globals() or "df_entrenamiento" not in globals(): return
    from src.property_text_pipeline import COLUMNA_OBJETIVO, COLUMNA_TEXTO_LIMPIO, COLUMNA_TEXTO_LIMPIO_CENSURADO, entrenar_modelo_base_svm, entrenar_modelo_bayes, entrenar_modelo_logistica
    if COLUMNA_OBJETIVO not in df_entrenamiento.columns or COLUMNA_OBJETIVO not in df_prueba.columns: return

    etiquetas_entrenamiento = df_entrenamiento[COLUMNA_OBJETIVO]

    def _entrenar_y_guardar(nombre_variable, funcion_entrenamiento, columna_texto):
        if globals().get(nombre_variable, None) is not None: return
        if columna_texto not in df_entrenamiento.columns or columna_texto not in df_prueba.columns: return
        modelo = funcion_entrenamiento(df_entrenamiento, etiquetas_entrenamiento, columna_texto=columna_texto)
        globals()[nombre_variable] = modelo.predict(df_prueba[columna_texto])

    _entrenar_y_guardar("predicciones_bayes", entrenar_modelo_bayes, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_svm", entrenar_modelo_base_svm, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_logistica", entrenar_modelo_logistica, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_bayes_censurado", entrenar_modelo_bayes, COLUMNA_TEXTO_LIMPIO_CENSURADO)
    _entrenar_y_guardar("predicciones_svm_censurado", entrenar_modelo_base_svm, COLUMNA_TEXTO_LIMPIO_CENSURADO)
    _entrenar_y_guardar("predicciones_logistica_censurado", entrenar_modelo_logistica, COLUMNA_TEXTO_LIMPIO_CENSURADO)

def buscar_prediccion(nombre):
    return globals().get(nombre, None)

y_real = df_prueba["property_type"] if "df_prueba" in globals() else None
if y_real is not None:
    _entrenar_modelos_clasicos_si_falta()

modelos = [
    ("Bayes", "Base", "predicciones_bayes"),
    ("SVM", "Base", "predicciones_svm"),
    ("Reg Log", "Base", "predicciones_logistica"),
    ("DistilBERT 1 epoch", "Base", None),
    ("Bayes", "Censurado", "predicciones_bayes_censurado"),
    ("SVM", "Censurado", "predicciones_svm_censurado"),
    ("Reg Log", "Censurado", "predicciones_logistica_censurado"),
    ("DistilBERT 1 epoch", "Censurado", None),
    ("DistilBERT 2 epoch", "Censurado", None),
]

tabla = []
for modelo, escenario, variable in modelos:
    y_pred = buscar_prediccion(variable) if variable else None
    tabla.append({
        "Modelo": modelo,
        "Escenario": escenario,
        **metricas_globales(y_real, y_pred, modelo, escenario),
    })

_df_tabla1 = pd.DataFrame(tabla)[["Modelo", "Escenario", "accuracy", "f1_macro"]]
df_tabla1 = _df_tabla1.rename(columns={"accuracy": "Accuracy global", "f1_macro": "F1 macro"})
print("Tabla 1: Resumen global de métricas")
display(df_tabla1)
if y_real is None:
    print("⚠️ df_prueba no está definido en el espacio de ejecución. Ejecuta previamente las celdas de carga y evaluación.")


Tabla 1: Resumen global de métricas


,Modelo,Escenario,Accuracy global,F1 macro
0,Bayes,Base,0.7757,0.6692
1,SVM,Base,0.8953,0.7925
2,Reg Log,Base,0.9040,0.8066
3,DistilBERT 1 epoch,Base,0.8897,0.7847
4,Bayes,Censurado,0.7503,0.6449
5,SVM,Censurado,0.8350,0.7205
6,Reg Log,Censurado,0.8457,0.7354
7,DistilBERT 1 epoch,Censurado,0.8163,0.6901
8,DistilBERT 2 epoch,Censurado,0.8353,0.7225


In [25]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

# Resultados extraídos directamente de la Fase 4
RESULTADOS_TRANSFORMER_CLASE = {
    "DistilBERT 1 epoch_Base": {"F1 Casa": 0.8646, "F1 Departamento": 0.9327, "F1 PH": 0.5569},
    "DistilBERT 1 epoch_Censurado": {"F1 Casa": 0.8223, "F1 Departamento": 0.8923, "F1 PH": 0.3556},
    "DistilBERT 2 epoch_Censurado": {"F1 Casa": 0.8507, "F1 Departamento": 0.8903, "F1 PH": 0.4265},
}

def f1_por_clase(y_true, y_pred, etiquetas, modelo, escenario):
    clave = f"{modelo}_{escenario}"
    if clave in RESULTADOS_TRANSFORMER_CLASE:
        return RESULTADOS_TRANSFORMER_CLASE[clave]
        
    if y_true is None or y_pred is None:
        return {f"F1 {etiq}": "N/A" for etiq in etiquetas}
        
    _, _, f1_scores, _ = precision_recall_fscore_support(y_true, y_pred, labels=etiquetas, average=None, zero_division=0)
    return {f"F1 {etiq}": round(float(v), 4) for etiq, v in zip(etiquetas, f1_scores)}

def _entrenar_modelos_clasicos_si_falta():
    if "df_prueba" not in globals() or "df_entrenamiento" not in globals(): return
    from src.property_text_pipeline import COLUMNA_OBJETIVO, COLUMNA_TEXTO_LIMPIO, COLUMNA_TEXTO_LIMPIO_CENSURADO, entrenar_modelo_base_svm, entrenar_modelo_bayes, entrenar_modelo_logistica
    if COLUMNA_OBJETIVO not in df_entrenamiento.columns or COLUMNA_OBJETIVO not in df_prueba.columns: return

    etiquetas_entrenamiento = df_entrenamiento[COLUMNA_OBJETIVO]

    def _entrenar_y_guardar(nombre_variable, funcion_entrenamiento, columna_texto):
        if globals().get(nombre_variable, None) is not None: return
        if columna_texto not in df_entrenamiento.columns or columna_texto not in df_prueba.columns: return
        modelo = funcion_entrenamiento(df_entrenamiento, etiquetas_entrenamiento, columna_texto=columna_texto)
        globals()[nombre_variable] = modelo.predict(df_prueba[columna_texto])

    _entrenar_y_guardar("predicciones_bayes", entrenar_modelo_bayes, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_svm", entrenar_modelo_base_svm, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_logistica", entrenar_modelo_logistica, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_bayes_censurado", entrenar_modelo_bayes, COLUMNA_TEXTO_LIMPIO_CENSURADO)
    _entrenar_y_guardar("predicciones_svm_censurado", entrenar_modelo_base_svm, COLUMNA_TEXTO_LIMPIO_CENSURADO)
    _entrenar_y_guardar("predicciones_logistica_censurado", entrenar_modelo_logistica, COLUMNA_TEXTO_LIMPIO_CENSURADO)

etiquetas_clase = ["Casa", "Departamento", "PH"]
y_real = df_prueba["property_type"] if "df_prueba" in globals() else None
if y_real is not None:
    _entrenar_modelos_clasicos_si_falta()

modelos_a_evaluar = [
    ("Bayes", "Base", "predicciones_bayes"),
    ("SVM", "Base", "predicciones_svm"),
    ("Reg Log", "Base", "predicciones_logistica"),
    ("DistilBERT 1 epoch", "Base", None),
    ("Bayes", "Censurado", "predicciones_bayes_censurado"),
    ("SVM", "Censurado", "predicciones_svm_censurado"),
    ("Reg Log", "Censurado", "predicciones_logistica_censurado"),
    ("DistilBERT 1 epoch", "Censurado", None),
    ("DistilBERT 2 epoch", "Censurado", None),
]

tabla = []
for modelo, escenario, variable in modelos_a_evaluar:
    y_pred = globals().get(variable, None) if variable else None
    clase_f1 = f1_por_clase(y_real, y_pred, etiquetas_clase, modelo, escenario)
    tabla.append({
        "Modelo": modelo,
        "Escenario": escenario,
        "F1 Casa": clase_f1.get("F1 Casa", "N/A"),
        "F1 Departamento": clase_f1.get("F1 Departamento", "N/A"),
        "F1 PH": clase_f1.get("F1 PH", "N/A"),
    })

_df_tabla2 = pd.DataFrame(tabla)[["Modelo", "Escenario", "F1 Casa", "F1 Departamento", "F1 PH"]]
print("Tabla 2: F1 por clase")
display(_df_tabla2)
if y_real is None:
    print("⚠️ df_prueba no está definido en el espacio de ejecución. Ejecuta previamente las celdas de carga y evaluación.")


Tabla 2: F1 por clase


,Modelo,Escenario,F1 Casa,F1 Departamento,F1 PH
0,Bayes,Base,0.8124,0.8633,0.3317
1,SVM,Base,0.8830,0.9520,0.5424
2,Reg Log,Base,0.8950,0.9551,0.5698
3,DistilBERT 1 epoch,Base,0.8646,0.9327,0.5569
4,Bayes,Censurado,0.7915,0.8419,0.3012
5,SVM,Censurado,0.8276,0.9145,0.4195
6,Reg Log,Censurado,0.8408,0.9196,0.4458
7,DistilBERT 1 epoch,Censurado,0.8223,0.8923,0.3556
8,DistilBERT 2 epoch,Censurado,0.8507,0.8903,0.4265


## Explicabilidad global de los modelos

En esta sección mostramos explicaciones globales para los tres modelos clásicos entrenados sobre datos censurados y para DistilBERT con los datos censurados del conjunto de prueba.


In [26]:
import numpy as np
import pandas as pd
from collections import defaultdict

from src.property_text_pipeline import (
    COLUMNA_TEXTO_LIMPIO_CENSURADO,
    entrenar_modelo_base_svm,
    entrenar_modelo_bayes,
    entrenar_modelo_logistica,
)

if "df_entrenamiento" not in globals():
    print("⚠️ No se encuentra df_entrenamiento. Ejecuta primero la celda de preparación de datos.")
else:
    df_train = df_entrenamiento
    y_train = df_train["property_type"]

    def _fit_si_falta(nombre, funcion, columna_texto):
        if globals().get(nombre) is not None:
            return globals()[nombre]
        if columna_texto not in df_train.columns:
            return None
        modelo = funcion(df_train, y_train, columna_texto=columna_texto)
        globals()[nombre] = modelo
        return modelo

    def _mostrar_coeficientes(modelo, nombre):
        if modelo is None:
            print(f"⚠️ No existe el modelo {nombre}.")
            return
        vectorizer = modelo.named_steps["tfidf"]
        clasificador = modelo.named_steps["clasificador"]
        vocab = vectorizer.get_feature_names_out()
        coefs = clasificador.coef_
        clases = clasificador.classes_
        tablas = []
        for idx, clase in enumerate(clases):
            top_idx = np.argsort(coefs[idx])[::-1][:15]
            top_features = vocab[top_idx]
            top_values = coefs[idx][top_idx]
            tablas.append(pd.DataFrame({
                "clase": clase,
                "palabra": top_features,
                "coeficiente": np.round(top_values, 4),
            }))
        resultado = pd.concat(tablas, ignore_index=True)
        display(resultado)

    modelo_reg_log_censurado = _fit_si_falta(
        "modelo_reg_log_censurado",
        entrenar_modelo_logistica,
        COLUMNA_TEXTO_LIMPIO_CENSURADO,
    )

    print("### Regresión Logística censurada: coeficientes por clase")
    _mostrar_coeficientes(modelo_reg_log_censurado, "Regresión Logística")


### Regresión Logística censurada: coeficientes por clase


,clase,palabra,coeficiente
0,Casa,lote,5.4797
1,Casa,chalet,3.8035
2,Casa,autos,3.1133
3,Casa,terreno,3.0007
4,Casa,galeria,2.6426
5,Casa,jardin,2.5835
6,Casa,garage,2.4453
7,Casa,fondo,2.0973
8,Casa,4_dormitorios,2.0517
9,Casa,barrio,1.9602


In [27]:
if "df_entrenamiento" in globals():
    modelo_svm_censurado = _fit_si_falta(
        "modelo_svm_censurado",
        entrenar_modelo_base_svm,
        COLUMNA_TEXTO_LIMPIO_CENSURADO,
    )
    modelo_bayes_censurado = _fit_si_falta(
        "modelo_bayes_censurado",
        entrenar_modelo_bayes,
        COLUMNA_TEXTO_LIMPIO_CENSURADO,
    )

    print("### SVM censurado: coeficientes por clase")
    _mostrar_coeficientes(modelo_svm_censurado, "SVM")


### SVM censurado: coeficientes por clase


,clase,palabra,coeficiente
0,Casa,lote,3.4635
1,Casa,chalet,3.1141
2,Casa,adicional,2.6808
3,Casa,terreno,2.0133
4,Casa,comparten,1.7515
5,Casa,autos,1.7164
6,Casa,escuelas,1.6760
7,Casa,niz,1.6667
8,Casa,trasera,1.6261
9,Casa,parrillero,1.5902


In [28]:
## EXPLICABILIDAD BAYES
import numpy as np
import pandas as pd

def _mostrar_palabras_discriminantes_bayes(modelo):
    if modelo is None:
        print("⚠️ No existe el modelo Bayes.")
        return

    vectorizer = modelo.named_steps["tfidf"]
    clasificador = modelo.named_steps["clasificador"]
    vocab = vectorizer.get_feature_names_out()
    log_prob = clasificador.feature_log_prob_
    clases = clasificador.classes_

    tablas = []
    for idx, clase in enumerate(clases):
        otros_idx = [i for i in range(len(clases)) if i != idx]
        max_log_prob_otros = np.max(log_prob[otros_idx], axis=0)
        diferencia = log_prob[idx] - max_log_prob_otros
        top_idx = np.argsort(diferencia)[::-1][:15]
        tablas.append(pd.DataFrame({
            "clase": clase,
            "palabra": vocab[top_idx],
            "poder_discriminante": np.round(diferencia[top_idx], 4),
            "log_prob_clase": np.round(log_prob[idx][top_idx], 4),
        }))
    resultado = pd.concat(tablas, ignore_index=True)
    display(resultado)

if 'modelo_bayes_censurado' in globals():
    print("### Naive Bayes Censurado: Palabras con mayor poder discriminante por clase")
    _mostrar_palabras_discriminantes_bayes(modelo_bayes_censurado)
elif 'modelo_bayes' in globals():
    print("### Naive Bayes: Palabras con mayor poder discriminante por clase")
    _mostrar_palabras_discriminantes_bayes(modelo_bayes)


### Naive Bayes Censurado: Palabras con mayor poder discriminante por clase


,clase,palabra,poder_discriminante,log_prob_clase
0,Casa,family,2.4781,-7.1338
1,Casa,laguna,2.4267,-6.7908
2,Casa,venecitas,2.2063,-7.8557
3,Casa,comparten,2.0485,-6.9337
4,Casa,filtro,2.0433,-7.8056
5,Casa,maroto,2.0263,-8.3152
6,Casa,villanueva,1.9940,-7.8408
7,Casa,piletero,1.9278,-8.4137
8,Casa,junior,1.9164,-8.0540
9,Casa,cerco,1.9148,-7.5023


In [ ]:
import pandas as pd
from collections import defaultdict
from transformers_interpret import SequenceClassificationExplainer

if "modelo_censurado" not in globals() or "df_prueba" not in globals() or "tokenizador" not in globals() or "id_a_etiqueta" not in globals():
    print("⚠️ Faltan variables necesarias para la explicabilidad. Asegúrate de que modelo_censurado, tokenizador, df_prueba e id_a_etiqueta existan.")
else:
    explainer = SequenceClassificationExplainer(
        modelo_censurado,
        tokenizador,
        custom_labels=id_a_etiqueta
    )

    importancia_por_clase = defaultdict(lambda: defaultdict(float))

    muestra = df_prueba.sample(n=100, random_state=42)

    for texto in muestra['texto_limpio']:
        tokens = tokenizador.encode(str(texto), add_special_tokens=False, truncation=True, max_length=500)
        texto_truncado = tokenizador.decode(tokens)
        atribuciones = explainer(texto_truncado)
        
        # Obtenemos el índice numérico seguro y lo mapeamos a su nombre real
        clase_predicha_idx = explainer.predicted_class_index
        clase_predicha = id_a_etiqueta[clase_predicha_idx]

        for palabra, score in atribuciones:
            if palabra not in tokenizador.all_special_tokens and len(palabra) > 2:
                # Solo sumamos los scores positivos (los que empujan hacia esa clase)
                if score > 0:
                    importancia_por_clase[clase_predicha][palabra] += score

    # Iteramos automáticamente sobre todas las clases encontradas
    for clase, importancias in importancia_por_clase.items():
        top_palabras = sorted(importancias.items(), key=lambda x: x[1], reverse=True)[:15]
        df_top = pd.DataFrame(top_palabras, columns=["Palabra", "Score Positivo Acumulado"])
        print(f"### Top 15 palabras representativas - {clase}")
        display(df_top)


### Top 15 palabras con mayor score acumulado - 0


,Palabra,Importancia Acumulada
0,lot,10.443937
1,planta,9.060024
2,casa,8.619381
3,jardin,6.478605
4,##eria,5.930742
5,auto,3.744277
6,##s,3.466235
7,propiedad,3.039360
8,##e,2.984276
9,gal,2.722987


### Top 15 palabras con mayor score acumulado - 1


,Palabra,Importancia Acumulada
0,bal,24.608342
1,edificio,15.238325
2,##con,7.062456
3,piso,4.626213
4,##s,4.453803
5,terra,2.414224
6,departamento,2.155104
7,##te,2.068008
8,torre,1.992205
9,##cion,1.974355


### Top 15 palabras con mayor score acumulado - 2


,Palabra,Importancia Acumulada
0,##llo,3.329744
1,bal,1.833920
2,independiente,1.689666
3,terra,1.017406
4,piso,1.010497
5,##con,0.770073
6,playa,0.751312
7,ramo,0.713043
8,##cale,0.556280
9,primer,0.552395
